# **Data Science and Machine Learning 1 - Year 4 Portfolio**
## Last.Fm API Data Pulling and Cleaning
*By Mark Lambert - C00192497*

*Lecturer: Ben O'Shaughnessey*

*Submission Date: 27th November 2026*

---

In third year I built and deployed a LastFm scrobbling program, which familiarised me with the LastFM API. 

I decided for my Data Science & Machine Learning Portfolio that using this data would be interesting and applicable.
        
This notebook focus' on connecting with the LastFm API using an API key, and then pulling scrobbles using the ```User.getRecentTracks``` and ```User.getLovedTracks``` endpoints.

The data returned requires processing and cleaning. This notebook also covers aggregating the data and merging with an external datasource to expand the features.

```%pip install pandas```

In [1]:
import requests
import pandas as pd
import lastfm_creds

API_KEY = lastfm_creds.API_KEY
USERNAME = lastfm_creds.USERNAME

In [2]:
all_tracks = []
page = 1
total_pages = 1

while page <= total_pages:
    response = requests.get('http://ws.audioscrobbler.com/2.0/', params={
        'method': 'user.getrecenttracks',
        'user': USERNAME,
        'api_key': API_KEY,
        'format': 'json',
        'limit': 200,
        'page': page
    })
    data = response.json()
    if page == 1:
        total_pages = int(data['recenttracks']['@attr']['totalPages'])
    all_tracks.extend(data['recenttracks']['track'])
    page += 1

The data's raw columns.

The ones that this portfolio will focus on are **artist, album, name and date**. ```@attr``` holds songs that are marked ```nowplaying = true```. 

As such, anything with this attribute will also be ignored later on.

In [3]:
scrobblesResponse = pd.DataFrame(all_tracks)

scrobblesResponse.columns

Index(['artist', 'streamable', 'image', 'mbid', 'album', 'name', 'url',
       'date'],
      dtype='str')

The raw data shows that the actual readable details for each artist, album are embedded with a ```#text``` key in a dictionary, which must be extracted

In [4]:
scrobblesResponse

,artist,streamable,image,mbid,album,name,url,date
0,{'mbid': '067102ea-9519-4622-9077-57ca4164cfbb...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",3ac21694-8885-3779-83c2-c745043758c9,{'mbid': '0c0f9ff9-0b07-4d12-8978-6522601eca9d...,Big Calm,https://www.last.fm/music/Morcheeba/_/Big+Calm,"{'uts': '1792268283', '#text': '17 Oct 2026, 2..."
1,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",,{'mbid': '00163fb1-5ab7-4fec-b0dd-a915e2722340...,Glory Box,https://www.last.fm/music/Portishead/_/Glory+Box,"{'uts': '1792268273', '#text': '17 Oct 2026, 2..."
2,{'mbid': '4bd95eea-b9f6-4d70-a36c-cfea77431553...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",16b6bb74-a655-3a40-98f9-cd8b4f1c64f2,{'mbid': '0cb9ccb0-487b-409b-992e-c41577b9a426...,Killer Is Me,https://www.last.fm/music/Alice+in+Chains/_/Ki...,"{'uts': '1792268261', '#text': '17 Oct 2026, 2..."
3,{'mbid': '3ae2fb37-8a23-429d-9920-bac811c4fc22...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",dba84167-c823-493a-aaa9-124d81eb68ce,{'mbid': 'cbe2c86a-a2b9-420d-906e-64d7ddf8f041...,Neighbors Blues,https://www.last.fm/music/Jack+White/_/Neighbo...,"{'uts': '1792268240', '#text': '17 Oct 2026, 2..."
4,{'mbid': '65f4f0c5-ef9e-490c-aee3-909e7ae6b2ab...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",657ee641-fcec-39ed-ab24-f5277334d1b6,{'mbid': '1dc3de77-4acc-40ec-ae07-48f4c45f48eb...,Helpless,https://www.last.fm/music/Metallica/_/Helpless,"{'uts': '1792268216', '#text': '17 Oct 2026, 2..."
...,...,...,...,...,...,...,...,...
18136,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",362c85ba-c79b-478e-8321-0383c3ac8101,{'mbid': '0d2cb66c-3a32-4f81-b977-9231c461c34a...,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,"{'uts': '1763463648', '#text': '18 Nov 2025, 1..."
18137,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",6f93589f-4a47-4a9d-9274-09a0f437ad15,{'mbid': '10ac58ca-0655-4b16-a6cb-58fdc309de0a...,Over,https://www.last.fm/music/Portishead/_/Over,"{'uts': '1763463370', '#text': '18 Nov 2025, 1..."
18138,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",6da0b598-d9cf-4f62-8376-bc3181e509af,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,"{'uts': '1763462844', '#text': '18 Nov 2025, 1..."
18139,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",ad79b526-8df8-4e79-8756-0607df153e59,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,"{'uts': '1763462559', '#text': '18 Nov 2025, 1..."


Next, as per [Carlos Medina's article](https://medium.com/hemisferios/using-the-last-fm-api-to-extract-data-and-analyzing-it-with-python-part-1-ffe96f812455), first we must select the colums of interest and, as the column name is obscure I have chosen to rename it to **song**

In [6]:
scrobbles = scrobblesResponse

scrobbles = scrobbles[['artist', 'album', 'name', 'url', 'date']]
scrobbles.rename(columns={'name': 'song'}, inplace=True)
scrobbles

,artist,album,song,url,date
0,{'mbid': 'c14b4180-dc87-481e-b17a-64e4150f90f6...,{'mbid': '4a71e6d7-8989-4e1b-b34f-2572330bb37c...,Deliverance,https://www.last.fm/music/Opeth/_/Deliverance,"{'uts': '1792160577', '#text': '16 Oct 2026, 1..."
1,{'mbid': 'f1571db1-c672-4a54-a2cf-aaa329f26f0b...,"{'mbid': '', '#text': 'To Record Only Water Fo...",Ramparts,https://www.last.fm/music/John+Frusciante/_/Ra...,"{'uts': '1792151805', '#text': '16 Oct 2026, 1..."
2,{'mbid': 'f1571db1-c672-4a54-a2cf-aaa329f26f0b...,"{'mbid': '', '#text': 'To Record Only Water Fo...",Fallout,https://www.last.fm/music/John+Frusciante/_/Fa...,"{'uts': '1792151674', '#text': '16 Oct 2026, 1..."
3,{'mbid': 'f1571db1-c672-4a54-a2cf-aaa329f26f0b...,"{'mbid': '', '#text': 'To Record Only Water Fo...",Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,"{'uts': '1792151488', '#text': '16 Oct 2026, 1..."
4,{'mbid': 'f1571db1-c672-4a54-a2cf-aaa329f26f0b...,"{'mbid': '', '#text': 'To Record Only Water Fo...",Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,"{'uts': '1792151485', '#text': '16 Oct 2026, 1..."
...,...,...,...,...,...
18076,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,{'mbid': '0d2cb66c-3a32-4f81-b977-9231c461c34a...,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,"{'uts': '1763463648', '#text': '18 Nov 2025, 1..."
18077,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,{'mbid': '10ac58ca-0655-4b16-a6cb-58fdc309de0a...,Over,https://www.last.fm/music/Portishead/_/Over,"{'uts': '1763463370', '#text': '18 Nov 2025, 1..."
18078,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,"{'uts': '1763462844', '#text': '18 Nov 2025, 1..."
18079,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,"{'uts': '1763462559', '#text': '18 Nov 2025, 1..."


Next, convert the ```artist``` ```album``` and ```date``` to strings

In [7]:
scrobbles['artist'] = scrobbles['artist'].astype(str)
scrobbles['album'] = scrobbles['album'].astype(str)
scrobbles['date'] = scrobbles['date'].astype(str)

Extract the embedded ```#text``` attribute from each of the ```artist, album``` and ```date``` fields

In [8]:
scrobbles[['mbid2','artist']]=scrobbles['artist'].str.split("#text':", n = 1, expand = True)
scrobbles= scrobbles.drop('mbid2', axis=1)
scrobbles[['mbid3','album']]=scrobbles['album'].str.split("#text':", n = 1, expand = True)
scrobbles= scrobbles.drop('mbid3', axis=1)
scrobbles[['uts','date']]=scrobbles['date'].str.split("#text':", n = 1, expand = True)
scrobbles.head()

,artist,album,song,url,date,uts
0,'Opeth'},'Deliverance'},Deliverance,https://www.last.fm/music/Opeth/_/Deliverance,"'16 Oct 2026, 14:22'}","{'uts': '1792160577', '"
1,'John Frusciante'},'To Record Only Water For Ten Days (U.S. Vers...,Ramparts,https://www.last.fm/music/John+Frusciante/_/Ra...,"'16 Oct 2026, 11:56'}","{'uts': '1792151805', '"
2,'John Frusciante'},'To Record Only Water For Ten Days (U.S. Vers...,Fallout,https://www.last.fm/music/John+Frusciante/_/Fa...,"'16 Oct 2026, 11:54'}","{'uts': '1792151674', '"
3,'John Frusciante'},'To Record Only Water For Ten Days (U.S. Vers...,Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,"'16 Oct 2026, 11:51'}","{'uts': '1792151488', '"
4,'John Frusciante'},'To Record Only Water For Ten Days (U.S. Vers...,Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,"'16 Oct 2026, 11:51'}","{'uts': '1792151485', '"


Removing the ```}``` and ```'``` characters from the ```artist and album``` entries

In [9]:
scrobbles['artist'] = scrobbles['artist'].str.strip("}'\" ")
scrobbles['album'] = scrobbles['album'].str.strip("}'\" ")
scrobbles.head()

,artist,album,song,url,date,uts
0,Opeth,Deliverance,Deliverance,https://www.last.fm/music/Opeth/_/Deliverance,"'16 Oct 2026, 14:22'}","{'uts': '1792160577', '"
1,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Ramparts,https://www.last.fm/music/John+Frusciante/_/Ra...,"'16 Oct 2026, 11:56'}","{'uts': '1792151805', '"
2,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Fallout,https://www.last.fm/music/John+Frusciante/_/Fa...,"'16 Oct 2026, 11:54'}","{'uts': '1792151674', '"
3,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,"'16 Oct 2026, 11:51'}","{'uts': '1792151488', '"
4,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,"'16 Oct 2026, 11:51'}","{'uts': '1792151485', '"


The ```date``` and ```uts``` fields also need to be stripped of the remaining dirty characters. I have also chosen to rename uts to **timestamp** as it is more descriptive.

In [10]:
scrobbles['date'] = scrobbles['date'].str.strip("}'\" ")
scrobbles['uts'] = scrobbles['uts'].str.strip("{uts':','\" ")
scrobbles.rename(columns={'uts': 'timestamp'}, inplace=True)
scrobbles.head()

,artist,album,song,url,date,timestamp
0,Opeth,Deliverance,Deliverance,https://www.last.fm/music/Opeth/_/Deliverance,"16 Oct 2026, 14:22",1792160577
1,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Ramparts,https://www.last.fm/music/John+Frusciante/_/Ra...,"16 Oct 2026, 11:56",1792151805
2,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Fallout,https://www.last.fm/music/John+Frusciante/_/Fa...,"16 Oct 2026, 11:54",1792151674
3,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,"16 Oct 2026, 11:51",1792151488
4,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,"16 Oct 2026, 11:51",1792151485


Next, the date field must be converted into a date only field

In [11]:
scrobbles['date'] = pd.to_datetime(scrobbles['date'], unit='s')
scrobbles['date_only'] = scrobbles['date'].dt.date
scrobbles

,artist,album,song,url,date,timestamp,date_only
0,Opeth,Deliverance,Deliverance,https://www.last.fm/music/Opeth/_/Deliverance,2026-10-16 14:22:00,1792160577,2026-10-16
1,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Ramparts,https://www.last.fm/music/John+Frusciante/_/Ra...,2026-10-16 11:56:00,1792151805,2026-10-16
2,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Fallout,https://www.last.fm/music/John+Frusciante/_/Fa...,2026-10-16 11:54:00,1792151674,2026-10-16
3,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,2026-10-16 11:51:00,1792151488,2026-10-16
4,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,2026-10-16 11:51:00,1792151485,2026-10-16
...,...,...,...,...,...,...,...
18076,Tame Impala,Lonerism,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,2025-11-18 11:00:00,1763463648,2025-11-18
18077,Portishead,Portishead,Over,https://www.last.fm/music/Portishead/_/Over,2025-11-18 10:56:00,1763463370,2025-11-18
18078,Tame Impala,Deadbeat,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,2025-11-18 10:47:00,1763462844,2025-11-18
18079,Tame Impala,Deadbeat,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,2025-11-18 10:42:00,1763462559,2025-11-18


Finally, the row which contains NaN must be removed, as this was a song that had the ```@attr``` of ```nowplaying = true```, so it should not count as a fully scrobbled song

In [12]:
scrobbles = scrobbles.dropna(subset=['date'])
scrobbles

,artist,album,song,url,date,timestamp,date_only
0,Opeth,Deliverance,Deliverance,https://www.last.fm/music/Opeth/_/Deliverance,2026-10-16 14:22:00,1792160577,2026-10-16
1,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Ramparts,https://www.last.fm/music/John+Frusciante/_/Ra...,2026-10-16 11:56:00,1792151805,2026-10-16
2,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Fallout,https://www.last.fm/music/John+Frusciante/_/Fa...,2026-10-16 11:54:00,1792151674,2026-10-16
3,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,2026-10-16 11:51:00,1792151488,2026-10-16
4,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,2026-10-16 11:51:00,1792151485,2026-10-16
...,...,...,...,...,...,...,...
18076,Tame Impala,Lonerism,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,2025-11-18 11:00:00,1763463648,2025-11-18
18077,Portishead,Portishead,Over,https://www.last.fm/music/Portishead/_/Over,2025-11-18 10:56:00,1763463370,2025-11-18
18078,Tame Impala,Deadbeat,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,2025-11-18 10:47:00,1763462844,2025-11-18
18079,Tame Impala,Deadbeat,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,2025-11-18 10:42:00,1763462559,2025-11-18


Then, three new columns- ```hour``` ```tod``` ```month``` are added. First, the time has to be converted to Dublin's time. Then, we can call the time_of_day function to classify between the four periods of a day: **morning, afternoon, evening and night.**

In [13]:
def time_of_day(hour):
    if hour >= 22 or hour < 5:
        return 'night'
    if hour < 12:
        return 'morning'
    if hour < 17:
        return 'afternoon'
    return 'evening'

local_time = scrobbles['date'].dt.tz_localize('UTC').dt.tz_convert('Europe/Dublin')
scrobbles['hour'] = local_time.dt.hour
scrobbles['tod'] = scrobbles['hour'].apply(time_of_day)
scrobbles['month'] = scrobbles['date'].dt.to_period('M')
scrobbles.head()

,artist,album,song,url,date,timestamp,date_only,hour,tod,month
0,Opeth,Deliverance,Deliverance,https://www.last.fm/music/Opeth/_/Deliverance,2026-10-16 14:22:00,1792160577,2026-10-16,15,afternoon,2026-10
1,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Ramparts,https://www.last.fm/music/John+Frusciante/_/Ra...,2026-10-16 11:56:00,1792151805,2026-10-16,12,afternoon,2026-10
2,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Fallout,https://www.last.fm/music/John+Frusciante/_/Fa...,2026-10-16 11:54:00,1792151674,2026-10-16,12,afternoon,2026-10
3,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,2026-10-16 11:51:00,1792151488,2026-10-16,12,afternoon,2026-10
4,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,2026-10-16 11:51:00,1792151485,2026-10-16,12,afternoon,2026-10


Aggregate the data into daily scrobbles

In [14]:
daily = scrobbles.groupby('date_only').size().reset_index(name='scrobble_count')
daily['date_only'] = pd.to_datetime(daily['date_only'])
daily.set_index('date_only', inplace=True)

In [15]:
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
for i in range(7):
    daily[days[i]] = (daily.index.dayofweek == i).astype(float)

Output the data to csv's so that I don't have to request the data each time I run the program

In [17]:
scrobbles.to_csv('datasets/scrobbles.csv', index=False)

In [18]:
daily.to_csv('datasets/daily.csv', index=True)

Using the [unofficial docs](https://lastfm-docs.github.io/api-docs/user/getLovedTracks/#examples) once again, the data can be combined with the ```User.getLovedTracks``` endpoint. A new field called ```is_loved``` can be joined to the full scrobble dataset

In [19]:
all_loved_tracks = []
loved_page = 1
total_loved_pages = 1

while loved_page <= total_loved_pages:
    response = requests.get('http://ws.audioscrobbler.com/2.0/', params={
        'method': 'user.getlovedtracks',
        'user': USERNAME,
        'api_key': API_KEY,
        'format': 'json',
        'limit': 200,
        'page': loved_page
    })
    data = response.json()
    if loved_page == 1:
        total_loved_pages = int(data['lovedtracks']['@attr']['totalPages'])
    tracks = data['lovedtracks']['track']
    for track in tracks:
        all_loved_tracks.append({
            'artist': track['artist']['name'],
            'track': track['name'],
            'timestamp': int(track['date']['uts']) if 'date' in track else None
        })
    loved_page += 1

Merging the scrobbles and retrieved lovedScrobbles tracks on the columns ```artist``` and ```song``` against ```track``` and adding the new col ```is_loved``` 

In [20]:
lovedScrobbles = pd.DataFrame(all_loved_tracks)
lovedScrobbles['is_loved'] = 1
lovedScrobbles = lovedScrobbles.drop_duplicates(subset=['artist', 'track'])

lovedMerged = pd.merge(
    scrobbles,
    lovedScrobbles[['artist', 'track', 'is_loved']],
    left_on=['artist', 'song'],
    right_on=['artist', 'track'],
    how='left'
)

loved = lovedMerged.drop(columns=['track'])
loved['is_loved'] = loved['is_loved'].fillna(0).astype(int)
loved

,artist,album,song,url,date,timestamp,date_only,hour,tod,month,is_loved
0,Opeth,Deliverance,Deliverance,https://www.last.fm/music/Opeth/_/Deliverance,2026-10-16 14:22:00,1792160577,2026-10-16,15,afternoon,2026-10,1
1,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Ramparts,https://www.last.fm/music/John+Frusciante/_/Ra...,2026-10-16 11:56:00,1792151805,2026-10-16,12,afternoon,2026-10,0
2,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Fallout,https://www.last.fm/music/John+Frusciante/_/Fa...,2026-10-16 11:54:00,1792151674,2026-10-16,12,afternoon,2026-10,0
3,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,2026-10-16 11:51:00,1792151488,2026-10-16,12,afternoon,2026-10,0
4,John Frusciante,To Record Only Water For Ten Days (U.S. Version),Away & Anywhere,https://www.last.fm/music/John+Frusciante/_/Aw...,2026-10-16 11:51:00,1792151485,2026-10-16,12,afternoon,2026-10,0
...,...,...,...,...,...,...,...,...,...,...,...
18076,Tame Impala,Lonerism,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,2025-11-18 11:00:00,1763463648,2025-11-18,11,morning,2025-11,1
18077,Portishead,Portishead,Over,https://www.last.fm/music/Portishead/_/Over,2025-11-18 10:56:00,1763463370,2025-11-18,10,morning,2025-11,1
18078,Tame Impala,Deadbeat,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,2025-11-18 10:47:00,1763462844,2025-11-18,10,morning,2025-11,1
18079,Tame Impala,Deadbeat,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,2025-11-18 10:42:00,1763462559,2025-11-18,10,morning,2025-11,0


In [21]:
loved.to_csv('datasets/scrobbles.csv', index=False)

In [5]:
spotify = pd.read_csv('datasets/spotify.csv')

In order to get more features from the data pulled from Last.Fm, I sourced a large dataset from [HuggingFace](https://huggingface.co/datasets/GD-Studio/embeat_45m_spotify_tracks). 

I had tried previously to use Kaggle spotify datasets, however I was getting a low match rate when merging with my scrobbled data. This dataset has over 45 million entries, and is over 12GB across 3 files. 

As such, the files had to be iterated through in chunks, to bypass malloc errors. 

In [1]:
import pandas as pd
import glob
import os

def clean_text(s):
    s = s.str.replace(r'\s*[\(\[].*?[\)\]]', '', regex=True) 
    s = s.str.replace(r'\s*-.*$', '', regex=True)
    s = s.str.replace(r'[^\w\s]', '', regex=True)
    return s.str.replace(r'\s+', ' ', regex=True).str.strip()

def clean_artist(s):
    s = s.str.lower().str.split('/').str[0].str.split('&').str[0]
    return s.str.replace(r'[^\w\s]', '', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()

loved = pd.read_csv('datasets/scrobbles.csv')
loved['clean_track'] = clean_text(loved['song'])
loved['clean_artist'] = clean_artist(loved['artist'])
loved_unique = loved.drop_duplicates(subset=['clean_artist', 'clean_track']).copy()

dataset_path = r"D:/dataset" 
parquet_files = glob.glob(os.path.join(dataset_path, "*.parquet"))

matched_data = []

for f in parquet_files:
    # load one file at a time
    data = pd.read_parquet(f)
    
    data['clean_track'] = clean_text(data['track_name'])
    data['clean_artist'] = clean_artist(data['artist_name'])
    
    # keep only items in loved_unique
    data = data[
        data['clean_artist'].isin(loved_unique['clean_artist']) & 
        data['clean_track'].isin(loved_unique['clean_track'])
    ]
    
    # if any matched data, add to matched_data
    if len(data) > 0:
        matched_data.append(data)

spotify = pd.concat(matched_data, ignore_index=True)
spotify = spotify.drop_duplicates(subset=['clean_artist', 'clean_track'])

final = pd.merge(
    loved_unique,
    spotify.drop(columns=['track_name', 'artist_name', 'album_name']),
    on=['clean_artist', 'clean_track'],
    how='left'
)

The final dataset now has 37 features

In [5]:
final.columns

Index(['artist', 'album', 'song', 'url', 'date', 'timestamp', 'date_only',
       'hour', 'tod', 'month', 'is_loved', 'clean_track', 'clean_artist',
       'track_id', 'isrc', 'popularity', 'explicit', 'artist_idx', 'artist_id',
       'artist_popularity', 'artist_genres', 'artist_genre_idx', 'album_id',
       'release_year', 'duration', 'time_signature', 'tempo', 'key', 'mode',
       'danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence'],
      dtype='str')

In [6]:
final = final.dropna()

In [7]:
final

,artist,album,song,url,date,timestamp,date_only,hour,tod,month,...,key,mode,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
1,Thin Lizzy,Bad Reputation,Southbound,https://www.last.fm/music/Thin+Lizzy/_/Southbound,2026-10-14 19:21:00,1792005674,2026-10-14,20,evening,2026-10,...,9.0,1.0,0.597,0.5740,-13.440000,0.0393,0.0012,0.0007,0.0555,0.7300
2,Alice in Chains,Unplugged,"Down In A Hole - Live at the Majestic Theatre,...",https://www.last.fm/music/Alice+in+Chains/_/Do...,2026-10-14 19:16:00,1792005366,2026-10-14,20,evening,2026-10,...,6.0,1.0,0.356,0.8180,-5.343000,0.0335,0.0122,0.0001,0.1230,0.3890
3,Mad Season,Above,Long Gone Day,https://www.last.fm/music/Mad+Season/_/Long+Go...,2026-10-14 19:10:00,1792005019,2026-10-14,20,evening,2026-10,...,9.0,1.0,0.672,0.3100,-15.011000,0.0299,0.9180,0.5130,0.1470,0.5200
4,Alice in Chains,Alice in Chains,Heaven Beside You,https://www.last.fm/music/Alice+in+Chains/_/He...,2026-10-14 19:05:00,1792004730,2026-10-14,20,evening,2026-10,...,10.0,0.0,0.551,0.7240,-7.090000,0.0320,0.0045,0.0002,0.1160,0.5590
5,Portishead,Roseland NYC Live 25 (Remastered 2023),Over - Live / Remastered 2023,https://www.last.fm/music/Portishead/_/Over+-+...,2026-10-14 19:00:00,1792004402,2026-10-14,20,evening,2026-10,...,1.0,1.0,0.632,0.7120,-9.439000,0.0434,0.4530,0.0002,0.1130,0.1750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2621,Aphex Twin,Drukqs,Petiatil Cx Htdui,https://www.last.fm/music/Aphex+Twin/_/Petiati...,2025-11-18 22:42:00,1763505754,2025-11-18,22,night,2025-11,...,0.0,0.0,0.421,0.0068,-29.681000,0.0391,0.9960,0.9560,0.1540,0.1390
2622,Autechre,Incunabula,Bike,https://www.last.fm/music/Autechre/_/Bike,2025-11-18 22:34:00,1763505273,2025-11-18,22,night,2025-11,...,5.0,1.0,0.751,0.6100,-14.946000,0.0430,0.0139,0.9140,0.0890,0.0375
2628,Pink Floyd,The Dark Side of the Moon,Speak to Me,https://www.last.fm/music/Pink+Floyd/_/Speak+t...,2025-11-18 20:59:00,1763499570,2025-11-18,20,evening,2025-11,...,1.0,1.0,0.592,0.0195,-33.349998,0.0369,0.3610,0.8390,0.1080,0.0313
2635,Eagles,Desperado,Certain Kind of Fool,https://www.last.fm/music/Eagles/_/Certain+Kin...,2025-11-18 15:13:00,1763478822,2025-11-18,15,afternoon,2025-11,...,2.0,1.0,0.659,0.6180,-7.551000,0.0328,0.4080,0.0000,0.3400,0.4670


In [8]:
final.to_csv('datasets/scrobbled_merged_with_hugging_face.csv', index=False)